# 作業 1：多隱藏層全連接神經網路 (DNN)

- 輸入層：6 個神經元
- 隱藏層 1：5 個神經元，使用 ReLU
- 隱藏層 2：4 個神經元，使用 ReLU
- 隱藏層 3：4 個神經元，使用 Sigmoid
- 輸出層：3 個神經元，使用 Softmax

我們會計算每一層的加權和 z、輸出 h、以及交叉熵誤差。


## 1. 網路架構與參數設定

此 DNN 的每層權重矩陣與偏值向量都由我們自行設定。
- x 的形狀：(6,)
- W1: (6, 5), b1: (5,)
- W2: (5, 4), b2: (4,)
- W3: (4, 4), b3: (4,)
- W4: (4, 3), b4: (3,)


定義每種不同算法的函式

In [7]:
import numpy as np

def relu(x):
    return np.maximum(0, x)

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def softmax(x):
    e_x = np.exp(x - np.max(x))
    return e_x / e_x.sum()

def cross_entropy_error(y, y_hat):
    eps = 1e-12
    return -np.sum(y * np.log(y_hat + eps))


設定每層使用的參數

In [8]:
x = np.array([0.1, -3.2, 1.4, 1.5, -0.2, 0.7], dtype=float)

W1 = np.array([
    [0.2, -0.1, 0.3, -0.2, 0.4],
    [-0.5, 0.6, 0.1, -0.1, 0.2],
    [0.1, -0.2, 0.4, 0.3, -0.3],
    [0.5, 0.2, -0.1, 0.6, -0.4],
    [-0.3, 0.4, 0.2, -0.5, 0.3],
    [0.7, -0.2, 0.5, 0.1, -0.3],
], dtype=float)
b1 = np.array([0.1, -0.2, 0.3, 0.2, -0.1], dtype=float)

W2 = np.array([
    [0.3, -0.2, 0.5, 0.1],
    [0.4, 0.2, -0.1, 0.3],
    [-0.3, 0.5, 0.2, -0.2],
    [0.1, -0.4, 0.6, 0.2],
    [0.5, 0.1, -0.2, 0.4],
], dtype=float)
b2 = np.array([0.2, -0.1, 0.4, 0.3], dtype=float)

W3 = np.array([
    [0.5, -0.2, 0.3, 0.1],
    [0.1, 0.4, -0.3, 0.2],
    [-0.1, 0.2, 0.5, -0.4],
    [0.3, -0.1, 0.2, 0.4],
], dtype=float)
b3 = np.array([-0.2, 0.1, 0.3, -0.1], dtype=float)

W4 = np.array([
    [0.2, -0.3, 0.5],
    [0.4, 0.1, -0.2],
    [-0.1, 0.3, 0.4],
    [0.5, -0.2, 0.1],
], dtype=float)
b4 = np.array([0.1, -0.2, 0.3], dtype=float)

print('x shape =', x.shape)
print('W1 shape =', W1.shape, ', W1 個數 =', W1.size, ', b1 個數 =', b1.size)
print('W2 shape =', W2.shape, ', W2 個數 =', W2.size, ', b2 個數 =', b2.size)
print('W3 shape =', W3.shape, ', W3 個數 =', W3.size, ', b3 個數 =', b3.size)
print('W4 shape =', W4.shape, ', W4 個數 =', W4.size, ', b4 個數 =', b4.size)


x shape = (6,)
W1 shape = (6, 5) , W1 個數 = 30 , b1 個數 = 5
W2 shape = (5, 4) , W2 個數 = 20 , b2 個數 = 4
W3 shape = (4, 4) , W3 個數 = 16 , b3 個數 = 4
W4 shape = (4, 3) , W4 個數 = 12 , b4 個數 = 3


In [9]:
z1 = x @ W1 + b1
h1 = relu(z1)

z2 = h1 @ W2 + b2
h2 = relu(z2)

z3 = h2 @ W3 + b3
h3 = sigmoid(z3)

z4 = h3 @ W4 + b4
y_hat = softmax(z4)

print('z1 shape =', z1.shape, ', z1 =', z1)
print('h1 shape =', h1.shape, ', h1 =', h1)
print('z2 shape =', z2.shape, ', z2 =', z2)
print('h2 shape =', h2.shape, ', h2 =', h2)
print('z3 shape =', z3.shape, ', z3 =', z3)
print('h3 shape =', h3.shape, ', h3 =', h3)
print('z4 shape =', z4.shape, ', z4 =', z4)
print('y_hat shape =', y_hat.shape, ', y_hat =', y_hat)


z1 shape = (5,) , z1 = [ 3.16 -2.33  0.73  1.99 -1.99]
h1 shape = (5,) , h1 = [3.16 0.   0.73 1.99 0.  ]
z2 shape = (4,) , z2 = [ 1.128 -1.163  3.32   0.868]
h2 shape = (4,) , h2 = [1.128 0.    3.32  0.868]
z3 shape = (4,) , z3 = [ 0.2924  0.4516  2.472  -0.968 ]
h3 shape = (4,) , h3 = [0.57258359 0.61101958 0.92215546 0.27527932]
z4 shape = (3,) , z4 = [ 0.50434867 -0.08908235  0.86047799]
y_hat shape = (3,) , y_hat = [0.3355456  0.185365   0.47908939]


## 2. 計算真實標籤與 Cross Entropy Error

假設 x 所對應的真實類別是第 3 類，因此標籤 y = [0, 0, 1]。
交叉熵誤差可用來衡量 y_hat 和 y 的差距。


In [10]:
y = np.array([0, 0, 1], dtype=float)
loss = cross_entropy_error(y, y_hat)
print('y =', y)
print('cross_entropy_error =', loss)
print('argmax(y_hat) + 1 =', np.argmax(y_hat) + 1)


y = [0. 0. 1.]
cross_entropy_error = 0.7358680739190562
argmax(y_hat) + 1 = 3


In [11]:
def dnn(x):
    x = np.asarray(x, dtype=float)
    z1 = x @ W1 + b1
    h1 = relu(z1)
    z2 = h1 @ W2 + b2
    h2 = relu(z2)
    z3 = h2 @ W3 + b3
    h3 = sigmoid(z3)
    z4 = h3 @ W4 + b4
    y_hat = softmax(z4)
    return y_hat

y_hat_2 = dnn(x)
print('dnn(x) =', y_hat_2)
print('是否與上一個結果相同？', np.allclose(y_hat, y_hat_2))
print('類別預測 =', np.argmax(y_hat_2) + 1)


dnn(x) = [0.3355456  0.185365   0.47908939]
是否與上一個結果相同？ True
類別預測 = 3


## 3. Gradio 製作互動式輸入介面

In [13]:
import numpy as np
import gradio as gr

def predict_class(x1, x2, x3, x4, x5, x6):
    x_input = np.array([x1, x2, x3, x4, x5, x6], dtype=float)
    y_hat = dnn(x_input)
    pred = int(np.argmax(y_hat) + 1)
    return f'類別 {pred}（機率最大）'

demo = gr.Interface(
    fn=predict_class,
    inputs=[
        gr.Number(label='x1'),
        gr.Number(label='x2'),
        gr.Number(label='x3'),
        gr.Number(label='x4'),
        gr.Number(label='x5'),
        gr.Number(label='x6'),
    ],
    outputs=gr.Textbox(label='預測結果'),
    title='DNN 類別預測器',
    description='輸入 6 個特徵值，輸出 y_hat 中機率最高的類別 1、2 或 3。'
)
demo.launch()


* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


Error while flagging: [WinError 2] 系統找不到指定的檔案。: '.gradio'
